In [6]:
%pip install -q sgp4 global-land-mask earthengine-api opencv-python pillow pandas


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datetime import datetime, timedelta, timezone
import math, json, os, shutil
from glob import glob
import numpy as np, pandas as pd
from PIL import Image

# ---- Earth Engine
PROJECT = "featuremappingdataset"

# ---- camera (Sony IMX252) and lenses: metres per pixel at ~420 km
W, H = 2064, 1544
LENSES = {"FOUND": 245, "LOST": 92}

# ---- dataset
OUT_DIR = "dataset"
SEASONS = {"jun-aug": ("2024-06-01", "2024-09-01"), "dec-feb": ("2023-12-01", "2024-03-01")}
N_PER_CLASS = {"land": 20, "coast": 20, "ocean": 10}
SIM_DAYS = 7          # days of orbit to pick capture points from
SEED = 0              # change for a different set of locations (e.g. a held-out test set)
MIN_SUN_ELEV = 10     # only daylight capture points
MIN_SPACING_KM = 600  # keeps frames from overlapping

# ---- test set export
EXPORT_DIR = "test_export"
MAP_SEASON, PHOTO_SEASON = "jun-aug", "dec-feb"   # stored map vs simulated new photo
ROT_DEG, SHIFT_PX = 5, (40, 25)                   # pose difference applied to every photo
CAMERA_EFFECTS = True                             # blur, vignetting, noise, JPEG on photos

In [ ]:
from sgp4.api import Satrec, WGS72, jday
from global_land_mask import globe

EPOCH = datetime(2026, 10, 28, 11, 12, 28, tzinfo=timezone.utc)
ELEMENTS = dict(inc=51.6335, raan=0.5562, ecc=0.0007038, argp=319.9799,
                ma=40.0670, revs_per_day=15.49088092, bstar=0.10526e-3)

def build_satellite():
    e = ELEMENTS
    sat = Satrec()
    days = (EPOCH - datetime(1949, 12, 31, tzinfo=timezone.utc)).total_seconds() / 86400
    sat.sgp4init(WGS72, "i", 25544, days, e["bstar"], 0.0, 0.0, e["ecc"],
                 math.radians(e["argp"]), math.radians(e["inc"]), math.radians(e["ma"]),
                 e["revs_per_day"] * 2 * math.pi / 1440, math.radians(e["raan"]))
    return sat

def propagate(start, hours, step_s):
    times = [start + timedelta(seconds=s) for s in range(0, int(hours * 3600) + 1, step_s)]
    jf = np.array([jday(t.year, t.month, t.day, t.hour, t.minute, t.second) for t in times])
    jd, fr = np.ascontiguousarray(jf[:, 0]), np.ascontiguousarray(jf[:, 1])
    err, r, _ = build_satellite().sgp4_array(jd, fr)
    tt = (jd + fr - 2451545.0) / 36525
    th = np.radians(((67310.54841 + (876600 * 3600 + 8640184.812866) * tt) % 86400) / 240)
    x = r[:, 0] * np.cos(th) + r[:, 1] * np.sin(th)
    y = -r[:, 0] * np.sin(th) + r[:, 1] * np.cos(th)
    lat = np.degrees(np.arctan2(r[:, 2], np.hypot(x, y)))
    lon = np.degrees(np.arctan2(y, x))
    return pd.DataFrame({"time": times, "lat": lat, "lon": lon})

def sun_elevation(lat, lon, times):
    doy = np.array([t.timetuple().tm_yday for t in times])
    hr = np.array([t.hour + t.minute / 60 + t.second / 3600 for t in times])
    g = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                    - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    ha = np.radians((hr * 60 + eqt + 4 * lon) / 4 - 180)
    phi = np.radians(lat)
    cz = np.sin(phi) * np.sin(decl) + np.cos(phi) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cz, -1, 1)))

def land_fraction(lat, lon, width_km, height_km, nx=11, ny=9):
    xs = np.linspace(-width_km / 2, width_km / 2, nx)
    ys = np.linspace(-height_km / 2, height_km / 2, ny)
    la = np.clip(lat[:, None, None] + ys[None, :, None] / 111.32, -89.9, 89.9)
    lo = lon[:, None, None] + xs[None, None, :] / (111.32 * np.cos(np.radians(lat))[:, None, None])
    return globe.is_land(la, (lo + 180) % 360 - 180).mean(axis=(1, 2))

def km_between(lat1, lon1, lat2, lon2):
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    return 6371 * np.arccos(np.clip(np.sin(p1) * np.sin(p2) + np.cos(p1) * np.cos(p2) * np.cos(dl), -1, 1))

track = propagate(EPOCH, SIM_DAYS * 24, 60)
track["sun_elev"] = sun_elevation(track.lat.values, track.lon.values, track.time)
big = LENSES["FOUND"] * W / 1000, LENSES["FOUND"] * H / 1000
track["land_frac"] = land_fraction(track.lat.values, track.lon.values, *big)
track["class_guess"] = np.select([track.land_frac < 0.02, track.land_frac > 0.9], ["ocean", "land"], "coast")
day = track[track.sun_elev >= MIN_SUN_ELEV]

picked = []
for cls, n in N_PER_CLASS.items():
    chosen = []
    for row in day[day.class_guess == cls].sample(frac=1, random_state=SEED).itertuples():
        if all(km_between(row.lat, row.lon, o.lat, o.lon) >= MIN_SPACING_KM for o in picked + chosen):
            chosen.append(row)
        if len(chosen) == n:
            break
    picked += chosen
    print(f"{cls}: {len(chosen)} points")
points = pd.DataFrame(picked).drop(columns="Index").reset_index(drop=True)
points.insert(0, "point_id", [f"p{i:03d}" for i in range(len(points))])
points

In [ ]:
import ee
ee.Initialize(project=PROJECT)

S2 = "COPERNICUS/S2_SR_HARMONIZED"
MODIS = "MODIS/061/MOD09A1"   # fills open ocean, which Sentinel-2 doesn't image

def utm_epsg(lat, lon):
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1

def frame_image(region, start, end):
    """Seasonal cloud-masked composite: RGB (uint8) + SCL labels."""
    col = (ee.ImageCollection(S2).filterBounds(region).filterDate(start, end)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40)))
    def clear(img):
        scl = img.select("SCL")
        ok = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
        return img.updateMask(ok).select(["B4", "B3", "B2"]).toFloat()
    # placeholders keep the bands in place when no Sentinel-2 scenes exist (open ocean)
    empty_rgb = ee.Image.constant([0, 0, 0]).rename(["B4", "B3", "B2"]).toFloat().updateMask(0)
    empty_scl = ee.Image.constant(0).rename("SCL").toUint8().updateMask(0)
    s2 = col.map(clear).merge(ee.ImageCollection([empty_rgb])).median()
    modis = (ee.ImageCollection(MODIS).filterDate(start, end)
             .select(["sur_refl_b01", "sur_refl_b04", "sur_refl_b03"], ["B4", "B3", "B2"])
             .median().toFloat())
    rgb = s2.unmask(modis).visualize(bands=["B4", "B3", "B2"], min=0, max=3000)
    scl = (col.select("SCL").map(lambda i: i.toUint8())
           .merge(ee.ImageCollection([empty_scl])).mode().unmask(0).toUint8().rename("SCL"))
    return rgb.addBands(scl)

def fetch(image, lat, lon, gsd, tiles=1):
    """Pixels on the camera grid (W x H, gsd metres, north-up), optionally in tiles x tiles pieces."""
    epsg = utm_epsg(lat, lon)
    cx, cy = ee.Geometry.Point([lon, lat]).transform(f"EPSG:{epsg}", 1).coordinates().getInfo()
    x0, y0 = cx - W / 2 * gsd, cy + H / 2 * gsd
    rows = []
    for r in np.array_split(np.arange(H), tiles):
        row = []
        for c in np.array_split(np.arange(W), tiles):
            row.append(ee.data.computePixels({
                "expression": image, "fileFormat": "NUMPY_NDARRAY",
                "grid": {"dimensions": {"width": len(c), "height": len(r)},
                         "affineTransform": {"scaleX": gsd, "shearX": 0, "translateX": x0 + c[0] * gsd,
                                             "shearY": 0, "scaleY": -gsd, "translateY": y0 - r[0] * gsd},
                         "crsCode": f"EPSG:{epsg}"}}))
        rows.append(np.concatenate(row, axis=1))
    return np.concatenate(rows, axis=0), epsg, [gsd, 0, x0, 0, -gsd, y0]

def fetch_retry(image, lat, lon, gsd):
    """Whole frame first; if Earth Engine runs out of memory, split into 2x2 then 4x4 tiles."""
    for tiles in (1, 2, 4):
        try:
            return fetch(image, lat, lon, gsd, tiles)
        except ee.EEException as e:
            if "memory" not in str(e).lower() and "timed out" not in str(e).lower():
                raise
    raise RuntimeError("still too large after splitting into 4x4 tiles")

print("Earth Engine ready")

In [ ]:
LAND_CLASSES = [2, 4, 5, 7, 11]      # SCL: dark area, vegetation, bare, unclassified, snow
CLOUD_CLASSES = [3, 8, 9, 10]        # SCL: cloud shadow, medium/high cloud, cirrus

def labels(scl):
    """Frame-level summary of the Sentinel-2 Scene Classification Layer."""
    n = scl.size
    covered = scl > 0                                  # 0 = no Sentinel-2 data (open ocean)
    land = np.isin(scl, LAND_CLASSES)
    water = (scl == 6) | ~covered
    clear = land.sum() + water.sum()
    land_share = land.sum() / clear if clear else 0.0
    out = {"s2_coverage": covered.mean(), "land_frac": land.sum() / n, "water_frac": water.sum() / n,
           "cloud_frac": np.isin(scl, CLOUD_CLASSES).sum() / n, "snow_frac": (scl == 11).sum() / n,
           "land_share_clear": land_share}
    out["class"] = ("ocean" if land_share < 0.02 and out["s2_coverage"] < 0.01
                    else "island" if land_share < 0.02
                    else "land" if land_share > 0.9 else "coast")
    return {k: (round(float(v), 4) if not isinstance(v, str) else v) for k, v in out.items()}

def save_frame(arr, meta):
    folder = os.path.join(OUT_DIR, meta["lens"])
    os.makedirs(folder, exist_ok=True)
    base = os.path.join(folder, f"{meta['point_id']}_{meta['season']}")
    rgb = np.dstack([arr["vis-red"], arr["vis-green"], arr["vis-blue"]]).astype(np.uint8)
    Image.fromarray(rgb).save(base + "_rgb.png")
    Image.fromarray(arr["SCL"].astype(np.uint8)).save(base + "_scl.png")
    with open(base + ".json", "w") as f:
        json.dump(meta, f, indent=2, default=str)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

jobs = [(p, lens, gsd, season, start, end)
        for p in points.itertuples() for lens, gsd in LENSES.items()
        for season, (start, end) in SEASONS.items()]
todo = [j for j in jobs
        if not os.path.exists(os.path.join(OUT_DIR, j[1], f"{j[0].point_id}_{j[3]}.json"))]
print(f"{len(jobs) - len(todo)} frames already saved, {len(todo)} to fetch")

def run(job):
    p, lens, gsd, season, start, end = job
    region = ee.Geometry.Point([p.lon, p.lat]).buffer(gsd * W * 0.75).bounds()
    arr, epsg, transform = fetch_retry(frame_image(region, start, end), p.lat, p.lon, gsd)
    meta = {"point_id": p.point_id, "lens": lens, "season": season, "gsd_m": gsd,
            "width": W, "height": H, "center_lat": p.lat, "center_lon": p.lon,
            "capture_time": p.time, "sun_elev": round(float(p.sun_elev), 1),
            "epsg": epsg, "transform": transform, **labels(arr["SCL"])}
    save_frame(arr, meta)
    return meta

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(run, j): j for j in todo}
    for i, f in enumerate(as_completed(futures), 1):
        p, lens, _, season, _, _ = futures[f]
        try:
            m = f.result()
            print(f"[{i}/{len(todo)}] {p.point_id} {lens} {season}: {m['class']}")
        except Exception as e:
            print(f"[{i}/{len(todo)}] {p.point_id} {lens} {season}: FAILED - {e}")

metas = pd.DataFrame([json.load(open(p)) for p in sorted(glob(os.path.join(OUT_DIR, "*", "*.json")))])
metas.to_csv(os.path.join(OUT_DIR, "index.csv"), index=False)
print(f"\n{len(metas)} frames in '{OUT_DIR}/' (re-run this cell to retry any failures)")
print(metas.groupby(["lens", "class"]).size().unstack(fill_value=0))

In [ ]:
import cv2

def camera_effects(img, seed=0):
    """Placeholder camera model: lens blur, darker corners, sensor noise, JPEG compression."""
    h, w = img.shape[:2]
    out = cv2.GaussianBlur(img.astype(np.float32), (0, 0), 0.8)
    yy, xx = np.mgrid[0:h, 0:w]
    out *= (1 - 0.3 * (((xx - w / 2) ** 2 + (yy - h / 2) ** 2) / ((w / 2) ** 2 + (h / 2) ** 2)))[..., None]
    out += np.random.default_rng(seed).normal(0, 3, out.shape)
    out = np.clip(out, 0, 255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", cv2.cvtColor(out, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 85])
    return cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

Hm = cv2.getRotationMatrix2D((W / 2, H / 2), ROT_DEG, 1.0)
Hm[:, 2] += SHIFT_PX
H_true = np.vstack([Hm, [0, 0, 1]])            # known map -> photo transform (the answer key)

metas = pd.read_csv(os.path.join(OUT_DIR, "index.csv"))
have = metas.groupby(["lens", "point_id"]).season.apply(set)
rows = []
for i, ((lens, pid), seasons) in enumerate(sorted(have.items())):
    if not {MAP_SEASON, PHOTO_SEASON} <= seasons:
        continue
    base = lambda s: os.path.join(OUT_DIR, lens, f"{pid}_{s}")
    ref = np.array(Image.open(base(MAP_SEASON) + "_rgb.png").convert("RGB"))
    photo = np.array(Image.open(base(PHOTO_SEASON) + "_rgb.png").convert("RGB"))
    photo = camera_effects(photo, seed=i) if CAMERA_EFFECTS else photo
    photo = cv2.warpPerspective(photo, H_true, (W, H))
    for kind, img, src in [("maps", ref, MAP_SEASON), ("photos", photo, PHOTO_SEASON)]:
        os.makedirs(os.path.join(EXPORT_DIR, lens, kind), exist_ok=True)
        Image.fromarray(img).save(os.path.join(EXPORT_DIR, lens, kind, f"{pid}.png"))
        shutil.copy(base(src) + "_scl.png", os.path.join(EXPORT_DIR, lens, kind, f"{pid}_scl.png"))
    m = metas[(metas.lens == lens) & (metas.point_id == pid) & (metas.season == PHOTO_SEASON)].iloc[0]
    rows.append({"lens": lens, "point_id": pid, "class": m["class"], "gsd_m": m.gsd_m,
                 "center_lat": m.center_lat, "center_lon": m.center_lon,
                 "map": f"{lens}/maps/{pid}.png", "photo": f"{lens}/photos/{pid}.png"})

pd.DataFrame(rows).to_csv(os.path.join(EXPORT_DIR, "index.csv"), index=False)
json.dump({"map_season": MAP_SEASON, "photo_season": PHOTO_SEASON, "rot_deg": ROT_DEG,
           "shift_px": SHIFT_PX, "camera_effects": CAMERA_EFFECTS, "seed": SEED,
           "H_true": H_true.tolist()},
          open(os.path.join(EXPORT_DIR, "settings.json"), "w"), indent=2)
zip_path = shutil.make_archive(EXPORT_DIR, "zip", EXPORT_DIR)
print(f"Exported {len(rows)} map/photo pairs to '{EXPORT_DIR}/' and '{zip_path}'")